
# 1. Imports & APIs
Importing window functions and SQL modules required for geographic ranking and statistical scoring.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 2. Geography Resolution
Loading Silver tables and resolving each unique customer to their most recent purchase location to ensure one row per person.

In [0]:
print("Loading Silver tables...")

transactions = spark.table("workspace.silver_marketing_project.silver_marketing_transactions")
customers = spark.table("workspace.silver_marketing_project.silver_customer_profiles")

# --- Geography Resolution ---
# A customer_unique_id may map to multiple customer_id (one per order),
# each potentially with different geography. For marketing, we resolve to
# the MOST RECENT purchase location (aligns with the Recency concept).

# Bring purchase timestamp into the customer/transaction join to rank by recency
cust_geo_ranked = (
    transactions.join(customers, on="customer_id", how="inner")
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "lat",
        "lng",
        "order_purchase_timestamp"
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_unique_id")
                  .orderBy(F.col("order_purchase_timestamp").desc())
        )
    )
    .filter(F.col("rn") == 1)   # keep only the most recent location per person
    .drop("rn", "order_purchase_timestamp")
)


# 3. RFM Metrics & Segmentation
Calculating Recency, Frequency, and Monetary metrics per unique customer, then applying quintile scoring and actionable marketing labels.

In [0]:
print("Calculating RFM metrics...")

max_date = transactions.select(F.max("order_purchase_timestamp")).collect()[0][0]
ref_date = max_date + F.expr("INTERVAL 1 DAY")

rfm_raw = (
    transactions.join(customers, on="customer_id", how="inner")
    .groupBy("customer_unique_id").agg(
        F.datediff(F.lit(ref_date), F.max("order_purchase_timestamp")).alias("recency_days"),
        F.count("order_id").alias("frequency_count"),
        F.sum("total_order_value").alias("monetary_sum")
    )
)

window = Window.partitionBy()
rfm_scores = rfm_raw.select("*",
    F.ntile(5).over(window.orderBy(F.col("recency_days").desc())).alias("R_Score"),
    F.ntile(5).over(window.orderBy("frequency_count")).alias("F_Score"),
    F.ntile(5).over(window.orderBy("monetary_sum")).alias("M_Score")
)

rfm_segmented = rfm_scores.withColumn("Marketing_Segment",
    F.when((F.col("R_Score") >= 4) & (F.col("F_Score") >= 4) & (F.col("M_Score") >= 4), "1. Champions")
     .when((F.col("R_Score") >= 3) & (F.col("F_Score") >= 3), "2. Loyal Customers")
     .when((F.col("R_Score") <= 2) & (F.col("F_Score") >= 4), "3. At Risk (Don't Lose Them)")
     .when((F.col("R_Score") <= 2) & (F.col("F_Score") <= 2), "5. Hibernating")
     .otherwise("4. Potential or Floating")
)


# 4. Building the Customer Dimension
Joining RFM segments with resolved geography to create dim_customers, one row per person, ready to filter the entire model.

In [0]:
print("Building gold_customers...")

gold_customers = rfm_segmented.join(cust_geo_ranked, on="customer_unique_id", how="left")

(gold_customers.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .option("comment", "DIMENSION: One row per unique customer. RFM scores, segment, and most-recent-purchase geography.")
  .saveAsTable("workspace.gold_marketing_project.gold_customers")
)
print("✓ gold_customers saved.")


# 5. Building the Orders Fact
Enriching transactions with the true customer identity (customer_unique_id) to create the central fact table of the star schema.

In [0]:
print("Building gold_orders...")

# Bring customer_unique_id into the fact via the customer_id bridge
gold_orders = (
    transactions.join(
        customers.select("customer_id", "customer_unique_id"),
        on="customer_id",
        how="inner"
    )
    .select(
        "order_id",
        "customer_unique_id",   # FK to dim_customers
        "order_purchase_timestamp",
        "total_order_value"
    )
)

(gold_orders.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .option("comment", "FACT: One row per delivered order. FK customer_unique_id links to dim_customers.")
  .saveAsTable("workspace.gold_marketing_project.gold_orders")
)
print("✓ gold_orders saved.")


# 6. Data Governance: Gold Columns
Documenting the fact table columns in the Unity Catalog so analysts understand the star schema keys and grain.

In [0]:
%sql
COMMENT ON COLUMN workspace.gold_marketing_project.gold_orders.order_id IS 'Unique identifier for the delivered order';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_orders.customer_unique_id IS 'Foreign key to dim_customers (the real person)';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_orders.order_purchase_timestamp IS 'Exact date and time the order was placed';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_orders.total_order_value IS 'Total monetary value paid per order';


# 7. Sanity Check
Validating that the Gold table preserves total revenue and row count from the Silver source (no data loss or duplication).

In [0]:
# Validation: gold_orders must match the Silver source exactly
silver_count = transactions.count()
gold_count = gold_orders.count()
silver_revenue = transactions.agg(F.sum("total_order_value")).collect()[0][0]
gold_revenue = gold_orders.agg(F.sum("total_order_value")).collect()[0][0]

print(f"Row count  — Silver: {silver_count:,} | Fact: {gold_count:,} | Match: {silver_count == gold_count}")
print(f"Revenue    — Silver: R$ {silver_revenue:,.2f} | Fact: R$ {gold_revenue:,.2f} | Match: {round(silver_revenue,2) == round(gold_revenue,2)}")